# Physics-Informed Graph Neural Network for River Modeling (v3 - Fixed)

**Key fixes in this version:**
1. **Curriculum learning**: Data-only training first (25 epochs), then gradually introduce physics loss
2. **No trivial collapse**: Physics loss starts at 0 so the model MUST learn the wave from data
3. **Start at hour 30**: Skips flat initialization period
4. **Cosine LR schedule**: Better convergence
5. **Removed teacher forcing**: Model always runs autoregressively within each unroll window

In [ ]:
!git clone https://github.com/ATIK2110018/gironde_PINN.git || (cd gironde_PINN && git pull origin main)
%cd gironde_PINN
!pip install -q torch-geometric netCDF4 xarray matplotlib scipy

### 1. Imports & Data Functions

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import MessagePassing
import netCDF4 as nc
import numpy as np
import math
from scipy.interpolate import griddata
from scipy.spatial import cKDTree
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def extract_graph_from_netcdf(nc_file_path):
    print(f"Reading mesh from {nc_file_path}...")
    dataset = nc.Dataset(nc_file_path, 'r')
    if 'mesh2d_node_x' in dataset.variables:
        node_x, node_y = dataset.variables['mesh2d_node_x'][:], dataset.variables['mesh2d_node_y'][:]
        edge_nodes = dataset.variables['mesh2d_edge_nodes'][:]
        node_z = dataset.variables['mesh2d_node_z'][:] if 'mesh2d_node_z' in dataset.variables else np.zeros_like(node_x)
    else:
        node_x, node_y = dataset.variables['NetNode_x'][:], dataset.variables['NetNode_y'][:]
        edge_nodes = dataset.variables['NetLink'][:]
        node_z = dataset.variables['NetNode_z'][:] if 'NetNode_z' in dataset.variables else np.zeros_like(node_x)
    is_spherical = (np.max(node_x) < 180.0) and (np.min(node_x) > -180.0)
    lat_to_m, lon_to_m = (111139.0, 111139.0 * math.cos(math.radians(np.mean(node_y)))) if is_spherical else (1.0, 1.0)
    edge_index_list, edge_attr_list = [], []
    for i in range(edge_nodes.shape[1] if edge_nodes.shape[0] == 2 else edge_nodes.shape[0]):
        n1, n2 = (int(edge_nodes[0, i]) - 1, int(edge_nodes[1, i]) - 1) if edge_nodes.shape[0] == 2 else (int(edge_nodes[i, 0]) - 1, int(edge_nodes[i, 1]) - 1)
        if n1 >= 0 and n2 >= 0:
            edge_index_list.extend([[n1, n2], [n2, n1]])
            dx_m, dy_m = (node_x[n2] - node_x[n1]) * lon_to_m, (node_y[n2] - node_y[n1]) * lat_to_m
            dist_m = math.sqrt(dx_m**2 + dy_m**2)
            edge_attr_list.extend([[dx_m, dy_m, dist_m], [-dx_m, -dy_m, dist_m]])
    dataset.close()
    return torch.tensor(np.column_stack((node_x, node_y)), dtype=torch.float32), torch.tensor(edge_index_list, dtype=torch.long).t().contiguous(), torch.tensor(edge_attr_list, dtype=torch.float32), torch.tensor(node_z, dtype=torch.float32)

def load_friction_xyz(filepath, node_coords):
    data = np.loadtxt(filepath)
    return torch.tensor(griddata((data[:, 0], data[:, 1]), data[:, 2], (node_coords[:, 0], node_coords[:, 1]), method='nearest'), dtype=torch.float32)

def load_boundary_pli(filepath, node_coords, threshold=0.002):
    with open(filepath, 'r') as f:
        lines = f.readlines()
    coords = []
    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) >= 2:
            try: coords.append([float(parts[0]), float(parts[1])])
            except ValueError: pass
    coords = np.array(coords)
    node_coords_np = node_coords.numpy()
    boundary_nodes = []
    for i in range(len(coords)-1):
        p1, p2 = coords[i], coords[i+1]
        l2 = np.sum((p2 - p1)**2)
        if l2 == 0: continue
        t = np.clip(np.sum((node_coords_np - p1) * (p2 - p1), axis=1) / l2, 0, 1)
        projection = p1 + t[:, np.newaxis] * (p2 - p1)
        dist = np.sqrt(np.sum((node_coords_np - projection)**2, axis=1))
        boundary_nodes.extend(np.where(dist < threshold)[0])
    return list(set(boundary_nodes))

def load_boundary_bc(filepath):
    times, values = [], []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 2:
                try: times.append(float(parts[0])); values.append(float(parts[1]))
                except ValueError: pass
    return np.array(times), np.array(values)

def load_truth_data(nc_file_path, node_coords):
    print(f"Loading truth data from {nc_file_path}...")
    dataset = nc.Dataset(nc_file_path, 'r')
    times = dataset.variables['time'][:]
    eta_raw = np.ma.filled(dataset.variables['mesh2d_s1'][:], 0.0)
    u_raw = np.ma.filled(dataset.variables['mesh2d_ucx'][:], 0.0)
    v_raw = np.ma.filled(dataset.variables['mesh2d_ucy'][:], 0.0)
    if eta_raw.shape[1] == node_coords.shape[0]:
        eta_nodes, u_nodes, v_nodes = eta_raw, u_raw, v_raw
    else:
        face_x = dataset.variables['mesh2d_face_x'][:]
        face_y = dataset.variables['mesh2d_face_y'][:]
        _, indices = cKDTree(np.column_stack((face_x, face_y))).query(node_coords.cpu().numpy())
        eta_nodes, u_nodes, v_nodes = eta_raw[:, indices], u_raw[:, indices], v_raw[:, indices]
    dataset.close()
    return times, torch.tensor(eta_nodes, dtype=torch.float32), torch.tensor(u_nodes, dtype=torch.float32), torch.tensor(v_nodes, dtype=torch.float32)

def interp_truth(t, true_times, true_data):
    """Interpolate truth data to arbitrary time t."""
    if t <= true_times[0]: return true_data[0]
    if t >= true_times[-1]: return true_data[-1]
    idx2 = np.searchsorted(true_times, t)
    idx1 = idx2 - 1
    w = (t - true_times[idx1]) / (true_times[idx2] - true_times[idx1] + 1e-8)
    return (1 - w) * true_data[idx1] + w * true_data[idx2]

def get_interp_val(t, times, values):
    return np.interp(t, times, values)

### 2. Model Architecture

In [ ]:
class HydroMPNNLayer(MessagePassing):
    def __init__(self, node_in_dim, edge_in_dim, out_dim):
        super().__init__(aggr='add')
        self.message_mlp = nn.Sequential(nn.Linear(node_in_dim * 2 + edge_in_dim, 128), nn.ReLU(), nn.Linear(128, 128))
        self.update_mlp = nn.Sequential(nn.Linear(node_in_dim + 128, 128), nn.ReLU(), nn.Linear(128, out_dim))
    
    def forward(self, x, edge_index, edge_attr):
        out = self.propagate(edge_index, x=x, edge_attr=edge_attr)
        return x + out if x.size(-1) == out.size(-1) else out
        
    def message(self, x_i, x_j, edge_attr):
        return self.message_mlp(torch.cat([x_i, x_j, edge_attr], dim=1))
    
    def update(self, aggr_out, x):
        return self.update_mlp(torch.cat([x, aggr_out], dim=1))

class RiverPIGNN(nn.Module):
    def __init__(self, node_features_dim=6, hidden_dim=128, edge_dim=3):
        super().__init__()
        self.encoder = nn.Linear(node_features_dim, hidden_dim)
        self.mpnn1 = HydroMPNNLayer(hidden_dim, edge_dim, hidden_dim)
        self.mpnn2 = HydroMPNNLayer(hidden_dim, edge_dim, hidden_dim)
        self.mpnn3 = HydroMPNNLayer(hidden_dim, edge_dim, hidden_dim)
        self.decoder = nn.Sequential(nn.Linear(hidden_dim, 64), nn.ReLU(), nn.Linear(64, 3))

    def forward(self, x, edge_index, edge_attr):
        h = F.relu(self.encoder(x))
        h = F.relu(self.mpnn1(h, edge_index, edge_attr))
        h = F.relu(self.mpnn2(h, edge_index, edge_attr))
        h = F.relu(self.mpnn3(h, edge_index, edge_attr))
        delta = self.decoder(h)
        
        # Use tanh to bound deltas, but allow larger range
        delta_eta = 1.0 * torch.tanh(delta[:, 0])  # +/- 1.0m per step
        delta_u   = 0.5 * torch.tanh(delta[:, 1])
        delta_v   = 0.5 * torch.tanh(delta[:, 2])
        
        out_eta = x[:, 0] + delta_eta
        out_u   = x[:, 1] + delta_u
        out_v   = x[:, 2] + delta_v
        zb = x[:, 3]
        
        # Soft clamping
        min_eta = zb + 0.05
        max_eta = zb + 25.0
        out_eta = torch.where(out_eta < min_eta, min_eta + 0.1 * (out_eta - min_eta), out_eta)
        out_eta = torch.where(out_eta > max_eta, max_eta + 0.1 * (out_eta - max_eta), out_eta)
        out_u = torch.clamp(out_u, -10.0, 10.0)
        out_v = torch.clamp(out_v, -10.0, 10.0)
        
        return torch.stack([out_eta, out_u, out_v], dim=1)

def physics_loss_swe(state_t, state_t_next, edge_index, edge_attr, dt=600.0, g=9.81):
    eta_t, u_t, v_t, zb, cf, _ = state_t.T
    h_t = torch.clamp(eta_t - zb, min=0.05)
    eta_next, u_next, v_next = state_t_next.T
    deta_dt = (eta_next - eta_t) / dt
    du_dt   = (u_next - u_t) / dt
    dv_dt   = (v_next - v_t) / dt
    
    row, col = edge_index
    dx, dy, dist = edge_attr.T
    dist_clamped = torch.clamp(dist, min=1.0)
    weight_x = dx / (dist_clamped**2 + 1e-8)
    weight_y = dy / (dist_clamped**2 + 1e-8)
    num_nodes = eta_t.size(0)
    
    def scatter(src, idx):
        return torch.zeros(num_nodes, dtype=src.dtype, device=src.device).scatter_add_(0, idx, src)
    
    d_eta = eta_t[col] - eta_t[row]
    d_hu  = (h_t[col]*u_t[col]) - (h_t[row]*u_t[row])
    d_hv  = (h_t[col]*v_t[col]) - (h_t[row]*v_t[row])
    d_u   = u_t[col] - u_t[row]
    d_v   = v_t[col] - v_t[row]
    
    grad_eta_x = scatter(d_eta * weight_x, row)
    grad_eta_y = scatter(d_eta * weight_y, row)
    grad_hu_x  = scatter(d_hu  * weight_x, row)
    grad_hv_y  = scatter(d_hv  * weight_y, row)
    grad_u_x   = scatter(d_u   * weight_x, row)
    grad_u_y   = scatter(d_u   * weight_y, row)
    grad_v_x   = scatter(d_v   * weight_x, row)
    grad_v_y   = scatter(d_v   * weight_y, row)
    
    vel_mag = torch.sqrt(u_t**2 + v_t**2 + 1e-8)
    mass = deta_dt + grad_hu_x + grad_hv_y
    mom_x = du_dt + u_t*grad_u_x + v_t*grad_u_y + g*grad_eta_x + cf*u_t*vel_mag/h_t
    mom_y = dv_dt + u_t*grad_v_x + v_t*grad_v_y + g*grad_eta_y + cf*v_t*vel_mag/h_t
    
    return torch.mean(mass**2) + torch.mean(mom_x**2) + torch.mean(mom_y**2)

### 3. Load Data

In [ ]:
node_coords, edge_index, edge_attr, node_z = extract_graph_from_netcdf("data/input/FlowFM_net.nc")
num_nodes = node_coords.size(0)
print(f"Graph: {num_nodes} nodes, {edge_index.size(1)} edges")

friction = load_friction_xyz("data/input/frictioncoefficient.xyz", node_coords)
bnd_port     = load_boundary_pli("data/input/port block.pli", node_coords, threshold=0.002)
bnd_garonne  = load_boundary_pli("data/input/garonne.pli",    node_coords, threshold=0.002)
bnd_dordogne = load_boundary_pli("data/input/dordogne.pli",   node_coords, threshold=0.002)
print(f"Boundary nodes: port={len(bnd_port)}, garonne={len(bnd_garonne)}, dordogne={len(bnd_dordogne)}")

t_port, v_port         = load_boundary_bc("data/input/WaterLevel.bc")
t_garonne, v_garonne   = load_boundary_bc("data/input/garonne.bc")
t_dordogne, v_dordogne = load_boundary_bc("data/input/dordogne.bc")

# Load truth data ONCE (not every epoch!)
map_files = [os.path.join(r, f) for d in ['/kaggle', '.'] if os.path.exists(d) for r, _, files in os.walk(d) for f in files if 'map.nc' in f.lower()]
has_truth_data = len(map_files) > 0
if has_truth_data:
    true_times, true_eta, true_u, true_v = load_truth_data(map_files[0], node_coords)
    print(f"Truth data: {len(true_times)} timesteps, range [{true_times[0]/3600:.1f}h - {true_times[-1]/3600:.1f}h]")
else:
    print("WARNING: No truth data found! Training with physics loss only.")

# Move to GPU
node_coords = node_coords.to(device)
edge_index  = edge_index.to(device)
edge_attr   = edge_attr.to(device)
node_z      = node_z.to(device)
friction    = friction.to(device)

### 4. Training with Curriculum Learning

**Phase 1 (epochs 1-25):** Data loss ONLY — forces the model to learn the tidal wave shape  
**Phase 2 (epochs 26-50):** Gradually ramp up physics loss as a regularizer

In [ ]:
model = RiverPIGNN(node_features_dim=6, hidden_dim=128, edge_dim=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-5)

# Training parameters
dt = 600.0
start_time = 30 * 3600.0   # Start at hour 30
max_time   = 72 * 3600.0   # End at hour 72
num_epochs = 50
unroll_steps = 6           # Shorter unroll = more frequent gradient updates

# Curriculum schedule
PHASE1_EPOCHS = 25  # Data-only phase

best_loss = float('inf')
loss_history = []

for epoch in range(num_epochs):
    # Curriculum: compute physics weight
    if epoch < PHASE1_EPOCHS:
        physics_weight = 0.0   # NO physics loss — learn the wave first!
        phase_str = "DATA-ONLY"
    else:
        # Linearly ramp physics from 0 to 0.1 over remaining epochs
        physics_weight = 0.1 * (epoch - PHASE1_EPOCHS) / (num_epochs - PHASE1_EPOCHS)
        phase_str = f"DATA+PHYS(w={physics_weight:.3f})"
    
    print(f"\n--- EPOCH {epoch+1}/{num_epochs} [{phase_str}] lr={optimizer.param_groups[0]['lr']:.2e} ---")
    
    current_time = start_time
    epoch_data_loss = 0.0
    epoch_phys_loss = 0.0
    epoch_steps = 0
    
    # Initialize state from TRUE data at start_time
    state_t = torch.zeros((num_nodes, 6), dtype=torch.float32, device=device)
    state_t[:, 3] = node_z
    state_t[:, 4] = friction
    if has_truth_data:
        init_eta = interp_truth(start_time, true_times, true_eta).to(device)
        init_u   = interp_truth(start_time, true_times, true_u).to(device)
        init_v   = interp_truth(start_time, true_times, true_v).to(device)
        state_t[:, 0] = init_eta
        state_t[:, 1] = init_u
        state_t[:, 2] = init_v
    else:
        state_t[:, 0] = torch.clamp(torch.tensor(0.0, device=device), min=node_z + 0.1)
    
    optimizer.zero_grad()
    accumulated_loss = 0.0
    step_count = 0
    
    while current_time < max_time:
        model.train()
        
        # Set boundary forcing
        state_t[:, 5] = 0.0
        state_t[bnd_port, 5]     = torch.tensor(get_interp_val(current_time, t_port, v_port), dtype=torch.float32, device=device).expand(len(bnd_port))
        state_t[bnd_garonne, 5]  = torch.tensor(get_interp_val(current_time, t_garonne, v_garonne) * 0.001, dtype=torch.float32, device=device).expand(len(bnd_garonne))
        state_t[bnd_dordogne, 5] = torch.tensor(get_interp_val(current_time, t_dordogne, v_dordogne) * 0.001, dtype=torch.float32, device=device).expand(len(bnd_dordogne))
        
        # Force boundary nodes to true water level
        if has_truth_data:
            curr_true_eta = interp_truth(current_time, true_times, true_eta).to(device)
            state_t[bnd_port, 0] = curr_true_eta[bnd_port]
        else:
            state_t[bnd_port, 0] = torch.tensor(get_interp_val(current_time, t_port, v_port), dtype=torch.float32, device=device).expand(len(bnd_port))
        
        # Forward pass
        predicted_next = model(state_t, edge_index, edge_attr)
        
        # Data loss (always active)
        if has_truth_data:
            target_eta = interp_truth(current_time + dt, true_times, true_eta).to(device)
            target_u   = interp_truth(current_time + dt, true_times, true_u).to(device)
            target_v   = interp_truth(current_time + dt, true_times, true_v).to(device)
            loss_data = F.mse_loss(predicted_next[:, 0], target_eta) + \
                        F.mse_loss(predicted_next[:, 1], target_u) + \
                        F.mse_loss(predicted_next[:, 2], target_v)
        else:
            loss_data = torch.tensor(0.0, device=device)
        
        # Physics loss (curriculum-controlled)
        if physics_weight > 0:
            loss_phys = physics_loss_swe(state_t, predicted_next, edge_index, edge_attr, dt=dt)
        else:
            loss_phys = torch.tensor(0.0, device=device)
        
        # Combined loss
        step_loss = loss_data + physics_weight * loss_phys
        accumulated_loss = accumulated_loss + step_loss
        
        epoch_data_loss += loss_data.item()
        epoch_phys_loss += loss_phys.item() if isinstance(loss_phys, torch.Tensor) and loss_phys.requires_grad else loss_phys if isinstance(loss_phys, (int, float)) else loss_phys.item()
        epoch_steps += 1
        step_count += 1
        
        # Log every 6 hours
        if current_time % (6*3600) == 0:
            print(f"  t={current_time/3600:.0f}h | Data MSE: {loss_data.item():.4f} | Phys: {loss_phys.item() if hasattr(loss_phys, 'item') else loss_phys:.4f}")
        
        # Truncated BPTT: backprop every unroll_steps
        if step_count % unroll_steps == 0:
            accumulated_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()
            accumulated_loss = 0.0
            
            # Reset state from TRUE data at this point (teacher forcing at segment boundaries)
            state_t = torch.zeros((num_nodes, 6), dtype=torch.float32, device=device)
            state_t[:, 3] = node_z
            state_t[:, 4] = friction
            if has_truth_data:
                state_t[:, 0] = target_eta.detach()
                state_t[:, 1] = target_u.detach()
                state_t[:, 2] = target_v.detach()
            else:
                state_t[:, :3] = predicted_next.detach().clone()
        else:
            # Within unroll window: use model's own predictions (autoregressive)
            new_state = torch.zeros((num_nodes, 6), dtype=torch.float32, device=device)
            new_state[:, :3] = predicted_next  # Keep grad for BPTT!
            new_state[:, 3] = node_z
            new_state[:, 4] = friction
            state_t = new_state
        
        current_time += dt
    
    # Flush remaining gradients
    if isinstance(accumulated_loss, torch.Tensor) and accumulated_loss.requires_grad:
        accumulated_loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    
    scheduler.step()
    
    avg_data = epoch_data_loss / max(epoch_steps, 1)
    avg_phys = epoch_phys_loss / max(epoch_steps, 1)
    loss_history.append(avg_data)
    
    print(f"  Epoch {epoch+1} Summary: Avg Data MSE = {avg_data:.4f} | Avg Phys = {avg_phys:.4f}")
    
    if avg_data < best_loss:
        best_loss = avg_data
        torch.save(model.state_dict(), "pignn_weights_best.pth")
        print(f"  --> NEW BEST MODEL! (Data MSE: {best_loss:.4f})")

print("\nTraining Complete!")

# Plot training curve
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(loss_history)+1), loss_history, 'b-o', markersize=3)
plt.axvline(x=PHASE1_EPOCHS, color='r', linestyle='--', label='Physics loss starts')
plt.xlabel('Epoch'); plt.ylabel('Avg Data MSE'); plt.title('Training Curve')
plt.legend(); plt.grid(True); plt.yscale('log')
plt.show()

### 5. Validation Plot

In [ ]:
node_idx = 5000
print(f"Loading best model and running inference for Node {node_idx}...")
model.load_state_dict(torch.load("pignn_weights_best.pth", map_location=device, weights_only=True))
model.eval()

# Initialize from true state at start_time
current_time = start_time
state_t = torch.zeros((num_nodes, 6), dtype=torch.float32, device=device)
state_t[:, 3] = node_z
state_t[:, 4] = friction
if has_truth_data:
    state_t[:, 0] = interp_truth(start_time, true_times, true_eta).to(device)
    state_t[:, 1] = interp_truth(start_time, true_times, true_u).to(device)
    state_t[:, 2] = interp_truth(start_time, true_times, true_v).to(device)
else:
    state_t[:, 0] = torch.clamp(torch.tensor(0.0, device=device), min=node_z + 0.1)

pred_times, pred_wl, true_wl_list = [], [], []

with torch.no_grad():
    while current_time < max_time:
        state_t[:, 5] = 0.0
        state_t[bnd_port, 5]     = torch.tensor(get_interp_val(current_time + dt, t_port, v_port), dtype=torch.float32, device=device).expand(len(bnd_port))
        state_t[bnd_garonne, 5]  = torch.tensor(get_interp_val(current_time + dt, t_garonne, v_garonne) * 0.001, dtype=torch.float32, device=device).expand(len(bnd_garonne))
        state_t[bnd_dordogne, 5] = torch.tensor(get_interp_val(current_time + dt, t_dordogne, v_dordogne) * 0.001, dtype=torch.float32, device=device).expand(len(bnd_dordogne))
        state_t[bnd_port, 0]     = torch.tensor(get_interp_val(current_time + dt, t_port, v_port), dtype=torch.float32, device=device).expand(len(bnd_port))
        
        state_t_next = model(state_t, edge_index, edge_attr)
        
        pred_times.append(current_time / 3600.0)
        pred_wl.append(state_t_next[node_idx, 0].item())
        
        if has_truth_data:
            true_eta_at_t = interp_truth(current_time + dt, true_times, true_eta)
            true_wl_list.append(true_eta_at_t[node_idx].item())
        
        state_t[:, :3] = state_t_next.clone()
        current_time += dt

print("Plotting...")
plt.figure(figsize=(14, 6))
if has_truth_data:
    plt.plot(pred_times, true_wl_list, 'b-', label='True Water Level (Delft3D)', linewidth=2)
plt.plot(pred_times, pred_wl, 'r--', label='Predicted Water Level (PIGNN)', linewidth=2)
plt.title(f"Water Level Comparison — Node {node_idx} (Hours {start_time/3600:.0f}–{max_time/3600:.0f})")
plt.xlabel("Time (hours)")
plt.ylabel("Water Level (m)")
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()